# Module 12 — Exercises

These exercises follow the order of the concept guide: the logic of a test and p-values, power and sample
size, t-tests (Welch and paired), proportions and chi-square, permutation tests, multiple comparisons,
experimental design, peeking, and finally comparing machine-learning models. Several checks compare your
answer against a *hidden* value, so compute carefully rather than guessing. `statsmodels` is not available
(and not needed): everything is NumPy plus `scipy.stats`.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb
from itertools import combinations
from scipy import stats
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Exact binomial p-value versus the normal approximation  `★☆☆`

Implement two functions for testing $H_0: \theta = \theta_0$ with $k$ successes in $n$ trials:

* `binom_two_sided_p(k, n, theta0=0.5)` — the exact two-sided p-value as the **doubled smaller tail**,
  $\min\big(1,\ 2\min(P(X \le k),\ P(X \ge k))\big)$ with $X \sim \text{Binomial}(n, \theta_0)$. Use
  `stats.binom.cdf` / `stats.binom.sf` (careful: `sf(k)` is $P(X > k)$, not $P(X \ge k)$). Return a float.
* `normal_approx_p(k, n, theta0=0.5, continuity=True)` — the two-sided normal-approximation p-value,
  $2\big(1 - \Phi(z)\big)$ with $z = \big(|k - n\theta_0| - \tfrac12\big)/\sqrt{n\theta_0(1-\theta_0)}$ when
  `continuity=True` (clip the numerator at 0) and without the $\tfrac12$ otherwise.

In [ ]:
def binom_two_sided_p(k, n, theta0=0.5):
    # YOUR CODE HERE
    return None

def normal_approx_p(k, n, theta0=0.5, continuity=True):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = binom_two_sided_p(50, 100)
except Exception:
    got = None
check_close('50 heads in 100 flips: p = 1', got, 1.0)
try:
    got = binom_two_sided_p(60, 100)
except Exception:
    got = None
check_hash('exact two-sided p for 60/100', got, '32cbcfcaa14e5d48', decimals=4)
try:
    got = binom_two_sided_p(40, 100)
except Exception:
    got = None
check_hash('symmetry: 40/100 gives the same p', got, '32cbcfcaa14e5d48', decimals=4)
try:
    got = normal_approx_p(60, 100, continuity=True)
except Exception:
    got = None
check_hash('normal approximation with continuity correction, 60/100', got, '81c19613769a0491', decimals=4)
try:
    got = normal_approx_p(60, 100, continuity=False)
except Exception:
    got = None
check_close('normal approximation without correction, 60/100', got, 2 * stats.norm.sf(2.0), tol=1e-6)
try:
    got = binom_two_sided_p(14, 30, 0.3)
except Exception:
    got = None
check_hash('asymmetric case: 14/30 under theta0 = 0.3', got, 'c535ca2d6acde81a', decimals=4)

## Exercise 2 — Power, required sample size and minimum detectable effect  `★☆☆`

For the two-sample z-test (known $\sigma$, $n$ per arm, two-sided level $\alpha$) implement

* `power_two_sample_z(delta, sigma, n, alpha=0.05)` — the power
  $\Phi\big(\tfrac{\delta}{\sigma\sqrt{2/n}} - z_{1-\alpha/2}\big) + \Phi\big(-\tfrac{\delta}{\sigma\sqrt{2/n}} - z_{1-\alpha/2}\big)$ as a float;
* `n_per_arm(delta, sigma, power=0.8, alpha=0.05)` — the required sample size per arm,
  $\big\lceil 2(z_{1-\alpha/2} + z_{1-\beta})^2\sigma^2/\delta^2 \big\rceil$, as an **int**;
* `mde(sigma, n, power=0.8, alpha=0.05)` — the minimum detectable effect $(z_{1-\alpha/2} + z_{1-\beta})\,\sigma\sqrt{2/n}$ as a float.

Use `stats.norm.ppf` for the quantiles and `stats.norm.cdf` / `stats.norm.sf` for $\Phi$.

In [ ]:
def power_two_sample_z(delta, sigma, n, alpha=0.05):
    # YOUR CODE HERE
    return None

def n_per_arm(delta, sigma, power=0.8, alpha=0.05):
    # YOUR CODE HERE
    return None

def mde(sigma, n, power=0.8, alpha=0.05):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = power_two_sample_z(0.0, 10.0, 50)
except Exception:
    got = None
check_close('no effect: power equals alpha = 0.05', got, 0.05, tol=1e-6)
try:
    got = power_two_sample_z(4.0, 10.0, 100)
except Exception:
    got = None
check_close('delta 4, sigma 10, n 100: power ~ 0.807', got, 0.8074, tol=1e-3)
try:
    got = power_two_sample_z(4.0, 10.0, 50)
except Exception:
    got = None
check_hash('power at n = 50', got, 'fda94e1202ba6f6b', decimals=3)
try:
    got = n_per_arm(4.0, 10.0)
    ok_int = isinstance(got, (int, np.integer))
except Exception:
    got, ok_int = None, False
check('n_per_arm returns an integer', ok_int)
check_hash('n per arm for delta 4, sigma 10, 80% power', got, '8c1f1046219ddd21')
try:
    got = n_per_arm(0.5, 2.0, power=0.9)
except Exception:
    got = None
check_hash('n per arm for delta 0.5, sigma 2, 90% power', got, '8ede6b26343305e0')
try:
    got = mde(10.0, 50)
except Exception:
    got = None
check_hash('MDE for sigma 10, n 50', got, '223623a06ccfbf48', decimals=3)
try:
    got = power_two_sample_z(mde(10.0, 50), 10.0, 50)
except Exception:
    got = None
check_close('consistency: power at the MDE is 0.8', got, 0.8, tol=1e-3)

## Exercise 3 — Welch's t-test from scratch  `★☆☆`

Implement `welch_t(x, y)` returning the tuple `(t, df, p)` for two independent samples:

$$t = \frac{\bar x - \bar y}{\sqrt{s_x^2/n_x + s_y^2/n_y}}, \qquad
\nu = \frac{(s_x^2/n_x + s_y^2/n_y)^2}{\dfrac{(s_x^2/n_x)^2}{n_x - 1} + \dfrac{(s_y^2/n_y)^2}{n_y - 1}}, \qquad
p = 2\,P(t_\nu \ge |t|).$$

Use sample variances with `ddof=1` and `stats.t.sf` for the tail probability. Do **not** call
`stats.ttest_ind` inside the function (the checks compare against it).

In [ ]:
def welch_t(x, y):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
old = np.array([52., 48, 55, 50, 53, 49, 51, 54]); new = np.array([47., 50, 44, 49, 46, 48])
ref = stats.ttest_ind(old, new, equal_var=False)
try:
    t, df, p = welch_t(old, new)
except Exception:
    t, df, p = None, None, None
check_close('cycle times: t statistic matches scipy', t, ref.statistic)
check_close('cycle times: Welch-Satterthwaite df matches scipy', df, ref.df)
check_close('cycle times: p-value matches scipy', p, ref.pvalue)
_r = np.random.default_rng(3)
x2 = _r.normal(0, 1, 12); y2 = _r.normal(0.5, 4, 40)          # very unequal variances and sizes
ref2 = stats.ttest_ind(x2, y2, equal_var=False)
try:
    t2, df2, p2 = welch_t(x2, y2)
except Exception:
    t2, df2, p2 = None, None, None
check_close('unequal variances: p-value matches scipy', p2, ref2.pvalue)
check('df lies between min(n) - 1 and n1 + n2 - 2', (df2 is not None) and (11 <= df2 <= 50))

## Exercise 4 — What a p-value is not  `★☆☆`

An A/B test on a new checkout page reports a lift of 0.6 percentage points with $p = 0.03$. For each of the
following statements say whether it is correct, and if not, what the correct statement is:

1. "There is a 3% probability that the new page is no better than the old one."
2. "There is a 97% probability that the new page is better."
3. "If the new page were truly no better, a lift at least this large (in either direction) would occur in about 3% of such experiments."
4. "If we repeated the experiment, it would come out significant 97% of the time."
5. "The result is more trustworthy than a competitor's $p = 0.04$ result from an experiment ten times larger."

<details><summary><b>Show answer</b></summary>

Only statement 3 is correct — it is the definition of a two-sided p-value: a probability about *data*
computed under $H_0$. Statement 1 reverses the conditioning: $P(\text{data} \mid H_0) \ne P(H_0 \mid \text{data})$;
the latter needs a prior over hypotheses and Bayes' theorem (with 10% of ideas being real improvements and
power 0.8, a significant result is a false positive about a third of the time). Statement 2 is the same error
for $H_1$. Statement 4 confuses the p-value with power: the replication probability depends on the true
effect and the sample size and, for an effect just large enough to give $p = 0.03$, is only around 50%.
Statement 5 is wrong twice: p-values from different sample sizes are not comparable as "trustworthiness",
and the larger experiment's $p = 0.04$ corresponds to a much *smaller* estimated effect with a much
*narrower* interval — compare effect sizes and confidence intervals, not p-values.

</details>

## Exercise 5 — Paired t-test and the cost of ignoring the pairing  `★★☆`

Six machines were timed before and after a maintenance change (minutes per cycle):

```
before = [31.0, 28.5, 35.2, 30.1, 27.8, 33.4]
after  = [29.1, 27.9, 33.0, 29.6, 27.0, 31.5]
```

1. Implement `paired_t(before, after)` returning `(t, df, p)` for the paired t-test on the differences
   $d_i = \text{before}_i - \text{after}_i$: $t = \bar d/(s_d/\sqrt n)$, $\nu = n - 1$ (an int), two-sided $p$.
   Do not call `stats.ttest_rel` inside the function.
2. Assign `t_paired, df_paired, p_paired = paired_t(before, after)`.
3. Assign `p_unpaired` = the two-sided p-value of **Welch's** test on the same two arrays (you may use
   `stats.ttest_ind(..., equal_var=False)` here), to see what ignoring the pairing costs.

In [ ]:
def paired_t(before, after):
    # YOUR CODE HERE
    return None

before = np.array([31.0, 28.5, 35.2, 30.1, 27.8, 33.4])
after = np.array([29.1, 27.9, 33.0, 29.6, 27.0, 31.5])
t_paired, df_paired, p_paired = None, None, None
p_unpaired = None
# YOUR CODE HERE

**Check your answer:**

In [ ]:
check_hash('paired t statistic', t_paired, '2dc1d1db43a8acf2', decimals=3)
check_hash('degrees of freedom', df_paired, 'ef2d127de37b942b')
check_hash('paired p-value', p_paired, '6e8117e55f1abda1', decimals=3)
try:
    got = paired_t(before, after)[2]
except Exception:
    got = None
check_close('matches stats.ttest_rel', got, stats.ttest_rel(before, after).pvalue)
check('unpaired Welch p-value is above 0.3 on the same numbers', (p_unpaired is not None) and p_unpaired > 0.3)
check('pairing makes the p-value more than 20 times smaller', (p_paired is not None and p_unpaired is not None) and p_unpaired / p_paired > 20)
try:
    got = paired_t(np.array([1., 2., 3., 4.]), np.array([0., 1., 2., 3.]) )
    ok = got is not None and np.isinf(got[0]) or (got is not None and abs(got[0]) > 1e6)
except Exception:
    ok = False
check('constant differences give an infinite (or huge) t', ok)

## Exercise 6 — Chi-square test of a 2 x 2 table and its identity with the z-test  `★★☆`

Implement

* `chi2_2x2(table)` — for a $2 \times 2$ array of counts, return `(chi2, p, expected)` where
  `expected[i, j] = R_i C_j / N` (row total times column total over the grand total),
  $\chi^2 = \sum_{ij}(O_{ij} - E_{ij})^2/E_{ij}$ **without** continuity correction, and $p = P(\chi^2_1 \ge \chi^2)$;
* `two_prop_z(x1, n1, x2, n2)` — the two-proportion z-statistic $(\hat\theta_2 - \hat\theta_1)/\sqrt{\bar\theta(1-\bar\theta)(1/n_1 + 1/n_2)}$ with the pooled rate $\bar\theta$.

Then convince yourself numerically that $\chi^2 = z^2$ for the conversion table `[[200, 3800], [240, 3760]]`
(rows = variants A and B, columns = converted / not converted).

In [ ]:
def chi2_2x2(table):
    # YOUR CODE HERE
    return None

def two_prop_z(x1, n1, x2, n2):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
conv = np.array([[200, 3800], [240, 3760]])
ref = stats.chi2_contingency(conv, correction=False)
try:
    chi2, p, E = chi2_2x2(conv)
except Exception:
    chi2, p, E = None, None, None
check_close('conversion table: chi2 matches scipy (no correction)', chi2, ref.statistic)
check_close('conversion table: p matches scipy', p, ref.pvalue)
check_close('expected counts', E, ref.expected_freq)
try:
    z = two_prop_z(200, 4000, 240, 4000)
except Exception:
    z = None
check_hash('two-proportion z for the conversion table', z, '32b357899515e9d2', decimals=3)
check_close('chi2 equals z squared', (z**2 if z is not None else None), chi2 if chi2 is not None else -1.0, tol=1e-9)
try:
    chi2s, ps, Es = chi2_2x2([[30, 170], [18, 182]])
except Exception:
    chi2s, ps = None, None
check_hash('side-effect table: chi2 statistic', chi2s, '8eae139162c567a7', decimals=3)
check_hash('side-effect table: p-value', ps, 'a79ac230af93369e', decimals=3)

## Exercise 7 — Permutation tests: exact and Monte Carlo  `★★☆`

Implement two permutation tests for the two-sided null "the two samples come from the same distribution",
using the absolute difference of means as the statistic:

* `exact_perm_p(a, b)` — enumerate **all** $\binom{n_a + n_b}{n_a}$ ways to choose which pooled observations
  form group $a$ (`itertools.combinations`), and return the fraction of splits whose
  $|\bar a^* - \bar b^*|$ is $\ge$ the observed value (use a tolerance such as `>= obs - 1e-12` to be safe with
  floating point). Only for tiny samples.
* `perm_test_mc(a, b, n_perm, rng)` — shuffle the pooled data `n_perm` times with `rng.permutation`, take the
  first $n_a$ values as the new group $a$, and return $\dfrac{1 + \#\{|T^*| \ge |t_{\text{obs}}|\}}{n_{\text{perm}} + 1}$.

Data for the Monte Carlo test (skewed spend per customer, 40 per arm) are generated below with a fixed seed.

```python
_r = np.random.default_rng(7)
spend_a = np.round(_r.exponential(20.0, 40), 2)          # control: skewed spend, mean 20
spend_b = np.round(_r.exponential(28.0, 40), 2)          # treatment: mean 28
```

In [ ]:
_r = np.random.default_rng(7)
spend_a = np.round(_r.exponential(20.0, 40), 2)          # control: skewed spend, mean 20
spend_b = np.round(_r.exponential(28.0, 40), 2)          # treatment: mean 28


def exact_perm_p(a, b):
    # YOUR CODE HERE
    return None

def perm_test_mc(a, b, n_perm, rng):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = exact_perm_p([12., 15, 11], [9., 8, 10])
except Exception:
    got = None
check_hash('exact two-sided p for the 3-vs-3 example', got, 'cc1a4126ba1a738f', decimals=3)
try:
    got = exact_perm_p([1., 2., 3.], [1., 2., 3.])
except Exception:
    got = None
check_close('identical groups: every split is at least as extreme, p = 1', got, 1.0)
try:
    got = exact_perm_p([5., 6., 7., 8.], [1., 2., 3., 4.])
except Exception:
    got = None
check_close('completely separated 4-vs-4 groups: p = 2/70', got, 2 / 70)
try:
    p_mc = perm_test_mc(spend_a, spend_b, 5_000, np.random.default_rng(1))
except Exception:
    p_mc = None
check('Monte Carlo p is a probability in (0, 1]', (p_mc is not None) and 0 < p_mc <= 1)
check('Monte Carlo p within 0.015 of the reference value', (p_mc is not None) and abs(p_mc - 0.0397) < 0.015,
      hint='did you compare |T*| with |t_obs| and add 1 to numerator and denominator?')
try:
    p_same = perm_test_mc(spend_a, spend_a, 2_000, np.random.default_rng(2))
except Exception:
    p_same = None
check('a sample against itself: p = 1', (p_same is not None) and abs(p_same - 1.0) < 1e-12)

## Exercise 8 — Bonferroni, Holm and Benjamini–Hochberg  `★★☆`

Implement three functions that take an array of (unsorted) p-values and return a **boolean array** of the
same length, `True` where the hypothesis is rejected:

* `bonferroni_reject(p, alpha=0.05)` — reject if $p_i \le \alpha/m$;
* `holm_reject(p, alpha=0.05)` — sort the p-values; compare $p_{(i)}$ with $\alpha/(m - i + 1)$ for
  $i = 1, 2, \dots$; reject all hypotheses **before the first failure** and nothing after it (step-down);
* `bh_reject(p, q=0.05)` — sort; find the **largest** $k$ with $p_{(k)} \le kq/m$; reject the $k$ hypotheses with
  the smallest p-values (step-up), or none if no such $k$ exists.

Remember to map the decisions back to the original order (use `np.argsort`). Apply them to the twelve
p-values below; the checks hash the sorted *indices* of the rejected hypotheses.

```python
pvals = np.array([0.040, 0.350, 0.001, 0.019, 0.600, 0.012, 0.200, 0.045, 0.011, 0.900, 0.004, 0.030])
```

In [ ]:
pvals = np.array([0.040, 0.350, 0.001, 0.019, 0.600, 0.012, 0.200, 0.045, 0.011, 0.900, 0.004, 0.030])


def bonferroni_reject(p, alpha=0.05):
    # YOUR CODE HERE
    return None

def holm_reject(p, alpha=0.05):
    # YOUR CODE HERE
    return None

def bh_reject(p, q=0.05):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
def _idx(fn, p, *args):
    try:
        r = np.asarray(fn(p, *args))
        return np.flatnonzero(r).tolist() if r.dtype == bool and r.shape == (len(p),) else None
    except Exception:
        return None
rb, rh, rq = _idx(bonferroni_reject, pvals), _idx(holm_reject, pvals), _idx(bh_reject, pvals)
check('functions return boolean arrays of the right length', rb is not None and rh is not None and rq is not None)
check_hash('Bonferroni rejections (indices)', rb, '1caa7b7fab4b45ed')
check_hash('Holm rejections (indices)', rh, '1caa7b7fab4b45ed')
check_hash('Benjamini-Hochberg rejections at q = 0.05 (indices)', rq, '71e0450c35e7d1fc')
check_hash('Benjamini-Hochberg rejections at q = 0.10 (indices)', _idx(bh_reject, pvals, 0.10), '2c0ad2173807f619')
check('Bonferroni is a subset of Holm, Holm a subset of BH', (rb is not None and rh is not None and rq is not None) and set(rb) <= set(rh) <= set(rq))
check('nothing rejected when all p-values are 0.5', _idx(bh_reject, np.full(5, 0.5)) == [] and _idx(holm_reject, np.full(5, 0.5)) == [])
check('everything rejected when all p-values are tiny', _idx(bonferroni_reject, np.full(6, 1e-6)) == [0, 1, 2, 3, 4, 5])

## Exercise 9 — Why randomise, and what blocking adds  `★★☆`

A streaming service lets users opt in to a redesigned home screen. After a month, opted-in users watch
on average 38 minutes per day and the others 52 minutes, so the product team concludes the redesign
*reduces* watch time. You notice that 80% of the opted-in users are on mobile, where average watch time is
30 minutes, while 80% of the others are on TVs, where it is 55 minutes; within each device type the
opted-in users actually watch about 3 minutes *more*.

1. Name the phenomenon and explain in two or three sentences why the overall comparison is misleading.
2. Explain why randomising the assignment (instead of letting users opt in) fixes the problem — including for
   confounders nobody has measured.
3. The team now plans a randomised test. What does *stratifying the randomisation by device* add on top of
   plain randomisation, and roughly when is it worth doing?
4. Name two further design safeguards you would put in the test plan and the failure each one prevents.

<details><summary><b>Show answer</b></summary>

1. This is Simpson's paradox caused by confounding: device type affects both who opts in (mobile users
   opt in far more) and the outcome (mobile sessions are shorter). The aggregate contrast compares a
   mostly-mobile treated group with a mostly-TV control group, so it measures the device effect, not the
   redesign; stratified by device, the redesign adds about 3 minutes.
2. Random assignment makes treatment independent of *every* user characteristic — device, age, tenure,
   enthusiasm, and whatever else nobody thought of — so in expectation the two groups have the same
   distribution of all covariates and the difference in means is unbiased for the average treatment effect.
   Adjusting for measured confounders in observational data can only fix the ones you measured.
3. Stratified (blocked) randomisation forces an exact 50:50 split *within* each device group, so the realised
   groups are balanced on device rather than balanced only on average, and the device-driven variance drops
   out of the comparison; the standard error shrinks by a factor involving the share of variance explained
   by the stratum (in the lesson, from about 0.25 to 0.15, like nearly tripling $n$). It is worth doing when
   a covariate is known in advance and strongly predicts the outcome; otherwise plain randomisation (or
   post-hoc covariate adjustment) does nearly as well.
4. Any two of: fix the sample size from a power calculation and a pre-registered primary metric (prevents
   peeking and metric shopping); an A/A test and a sample-ratio-mismatch check (prevents a broken randomiser
   or logging from masquerading as an effect); run for whole weekly cycles (prevents novelty and day-of-week
   effects); randomise by user rather than by session and watch for interference, e.g. shared household
   accounts (prevents contamination between arms); Holm or BH adjustment for guardrail metrics (prevents
   one chance "significant" guardrail out of many from driving the decision).

</details>

## Exercise 10 — How much does peeking inflate the false-positive rate?  `★★★`

Implement `peeking_fpr(n_looks, n_per_look, n_sims, rng, alpha=0.05, z_crit=None)` that simulates an A/B test
**under the null** (both arms standard normal, known $\sigma = 1$) analysed sequentially:

* each simulated experiment collects `n_per_look` new observations per arm at each of `n_looks` looks;
* at look $k$ (cumulative $n_k = k \cdot$ `n_per_look` per arm) the statistic is
  $Z_k = (\bar b_k - \bar a_k)/\sqrt{2/n_k}$, equivalently $(\text{sum}_b - \text{sum}_a)/\sqrt{2n_k}$;
* the experiment "stops with a significant result" if $|Z_k| > z_c$ at **any** look, where
  $z_c = z_{1-\alpha/2}$ by default or the supplied `z_crit`;
* return the fraction of the `n_sims` experiments that stopped with a significant result (a float).

Vectorise: draw arrays of shape `(n_sims, n_looks, n_per_look)`, sum over the last axis, take cumulative sums
along the look axis. Use the `rng` passed in, not the global one.

In [ ]:
def peeking_fpr(n_looks, n_per_look, n_sims, rng, alpha=0.05, z_crit=None):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    f1 = peeking_fpr(1, 1000, 4000, np.random.default_rng(10))
except Exception:
    f1 = None
check('one look: false-positive rate close to 0.05', (f1 is not None) and 0.035 < f1 < 0.065)
try:
    f5 = peeking_fpr(5, 200, 4000, np.random.default_rng(11))
except Exception:
    f5 = None
check('five looks: rate between 0.11 and 0.18 (Armitage: 0.142)', (f5 is not None) and 0.11 < f5 < 0.18)
try:
    f20 = peeking_fpr(20, 50, 4000, np.random.default_rng(12))
except Exception:
    f20 = None
check('twenty looks: rate between 0.20 and 0.30 (Armitage: 0.248)', (f20 is not None) and 0.20 < f20 < 0.30)
check('more looks, more false positives', (f1 is not None and f5 is not None and f20 is not None) and f1 < f5 < f20)
try:
    fp = peeking_fpr(5, 200, 4000, np.random.default_rng(13), z_crit=2.413)
except Exception:
    fp = None
check("Pocock's boundary 2.413 for 5 looks restores roughly 0.05", (fp is not None) and 0.03 < fp < 0.075)
try:
    f_strict = peeking_fpr(5, 200, 4000, np.random.default_rng(14), alpha=0.01)
except Exception:
    f_strict = None
check('alpha = 0.01 with five looks: rate between 0.02 and 0.05', (f_strict is not None) and 0.02 < f_strict < 0.05)

## Exercise 11 — McNemar's test and the Nadeau–Bengio correction  `★★★`

Two ways to compare classifiers honestly.

1. `mcnemar(y_true, pred_a, pred_b)` → `(b, c, chi2_cc, p_exact)` for two classifiers evaluated on the **same**
   test set: `b` = number of examples A gets right and B wrong, `c` = the reverse (both ints),
   `chi2_cc` $= (|b - c| - 1)^2/(b + c)$ (the continuity-corrected statistic), and `p_exact` = the exact
   two-sided p-value of `stats.binomtest(min(b, c), b + c, 0.5)`.
2. `nadeau_bengio_t(diffs, n_test, n_train)` → `(t, p)`: for $J$ paired per-fold score differences $d_j$,
   $t = \bar d \big/ \sqrt{(1/J + n_{\text{test}}/n_{\text{train}})\,s_d^2}$ with $s_d^2$ the sample variance
   (`ddof=1`), and $p = 2P(t_{J-1} \ge |t|)$.

The test data below are generated with a fixed seed: 800 test examples, and 50 fold differences from a
$10 \times 5$-fold cross-validation (so `n_test = 100`, `n_train = 400` for a dataset of 500 examples).

```python
_r = np.random.default_rng(11)
y_true = _r.integers(0, 2, 800)
right_a = _r.random(800) < 0.90                                   # classifier A is right 90% of the time
right_b = np.where(right_a, _r.random(800) < 0.94, _r.random(800) < 0.45)   # B tends to be right when A is
pred_a = np.where(right_a, y_true, 1 - y_true)
pred_b = np.where(right_b, y_true, 1 - y_true)
cv_diffs = np.round(_r.normal(0.006, 0.015, 50), 4)             # 50 paired fold differences, 10 x 5-fold CV
```

In [ ]:
_r = np.random.default_rng(11)
y_true = _r.integers(0, 2, 800)
right_a = _r.random(800) < 0.90                                   # classifier A is right 90% of the time
right_b = np.where(right_a, _r.random(800) < 0.94, _r.random(800) < 0.45)   # B tends to be right when A is
pred_a = np.where(right_a, y_true, 1 - y_true)
pred_b = np.where(right_b, y_true, 1 - y_true)
cv_diffs = np.round(_r.normal(0.006, 0.015, 50), 4)             # 50 paired fold differences, 10 x 5-fold CV


def mcnemar(y_true, pred_a, pred_b):
    # YOUR CODE HERE
    return None

def nadeau_bengio_t(diffs, n_test, n_train):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    b, c, chi2_cc, p_exact = mcnemar(y_true, pred_a, pred_b)
except Exception:
    b, c, chi2_cc, p_exact = None, None, None, None
check_hash('discordant count b (A right, B wrong)', b, 'aea92132c4cbeb26')
check_hash('discordant count c (B right, A wrong)', c, '624b60c58c9d8bfb')
check_hash('continuity-corrected McNemar statistic', chi2_cc, '5ecf536a3b7ecd85', decimals=3)
check_hash('exact McNemar p-value', p_exact, '575dc2e1f9814480', decimals=3)
try:
    got = mcnemar(np.array([0, 1, 0, 1, 1, 0]), np.array([0, 1, 0, 1, 0, 1]), np.array([0, 1, 1, 1, 1, 0]))
except Exception:
    got = None
check_close('tiny example: b = 1, c = 2, chi2 = 0', (got[:3] if got is not None else None), (1, 2, 0.0))
try:
    t_nb, p_nb = nadeau_bengio_t(cv_diffs, 100, 400)
except Exception:
    t_nb, p_nb = None, None
check_hash('Nadeau-Bengio t statistic', t_nb, '1bec39b32c2e3238', decimals=3)
check_hash('Nadeau-Bengio p-value', p_nb, 'ff58cf13d4266285', decimals=3)
try:
    t_naive_like, _ = nadeau_bengio_t(cv_diffs, 0, 400)        # n_test = 0 switches the correction off
except Exception:
    t_naive_like = None
check_close('with n_test = 0 the statistic is the naive paired t', t_naive_like, stats.ttest_1samp(cv_diffs, 0).statistic)
check('the correction shrinks |t| by a factor between 3 and 4 here', (t_nb is not None and t_naive_like is not None) and 3 < abs(t_naive_like / t_nb) < 4)

## Exercise 12 — Mini-project: design and analyse an A/B test end to end  `★★★`

You are the analyst for a checkout redesign. Baseline conversion is 5.0%; the smallest lift worth shipping
is a **10% relative** improvement; the site has 5,000 visitors a day. Use `rng = np.random.default_rng(2024)`
and the two-proportion z-test throughout.

1. **Design.** Compute the minimum detectable absolute effect, the required $n$ per arm for $\alpha = 0.05$
   (two-sided) and power 0.8 using the proportion version of the sample-size formula,
   $n = (z_{1-\alpha/2} + z_{1-\beta})^2\,[\theta_1(1-\theta_1) + \theta_2(1-\theta_2)]/\delta^2$, and the number of
   days the test must run.
2. **Run and analyse.** Simulate the experiment once with a true 10% relative lift (`rng.binomial` for each
   arm). Report the z-statistic, the p-value, the absolute and relative lift with a 95% confidence interval
   (unpooled standard error) and a ship / do-not-ship decision.
3. **Validate the design.** Simulate 4,000 experiments at the minimum detectable effect and 4,000 under $H_0$;
   report the empirical power and false-positive rate.
4. **Peeking.** Split the run into 14 daily batches. Under $H_0$, simulate 3,000 experiments that check the
   z-test every day and stop at the first $p < 0.05$: report the false-positive rate, the median stopping day
   and the mean absolute "lift" reported by the stopped experiments. Under the true lift, report how often a
   peeking analyst would stop before day 14 and how inflated the estimated lift is at those early stops.
5. Plot the running z-statistic of a few null experiments with the $\pm 1.96$ lines.

**Rubric — you have mastered this when you can:**

* The sample size comes from the formula with the correct $z$ quantiles ($1.96$ and $0.84$) and is rounded up; the duration follows from it (about 12.5 days, so plan two full weeks).
* The analysis reports effect size **and** interval **and** p-value, and the decision rule is stated before the result is looked at.
* The validation simulation is vectorised (arrays of binomial counts, not Python loops over visitors) and reproduces power $\approx 0.8$ and false-positive rate $\approx 0.05$.
* The peeking simulation uses cumulative counts per day and the pooled standard error at each look; it shows a false-positive rate near 0.2 for 14 looks and an inflated effect estimate at early stops under the alternative.
* The figure has a title, axis labels and the $\pm 1.96$ boundaries.
* The write-up draws the right conclusion: fix the horizon in advance (or use a sequential boundary) and report the interval rather than a verdict.

In [ ]:
# Your mini-project workspace

## Score

In [ ]:
summary()